# 입찰메이트 RFP RAG — 라이브 데모 (GCP VM · 자체호스팅 EXAONE)

이 노트북은 **VM에서 상시 구동 중인 FastAPI(`localhost:8500`)** 를 호출해
**자체호스팅 EXAONE-3.5-7.8B + bge-m3 하이브리드 검색**을 실시간으로 시연한다.

- 검색·생성 모두 VM 내부에서 돌아가므로 터널·방화벽 불필요.
- 사전: VM에서 API가 떠 있어야 함 — `nohup uvicorn src.api.main:app --host 127.0.0.1 --port 8500 & disown`

In [ ]:
# 1) API 클라이언트 (표준 urllib만 사용 — 설치 불필요)
import json, urllib.request

API = "http://localhost:8500"  # VM 상시 API

def _post(path, payload):
    req = urllib.request.Request(
        API + path, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=180) as r:
        return json.load(r)

def _get(path):
    with urllib.request.urlopen(API + path, timeout=60) as r:
        return json.load(r)

print("health:", _get("/health"))  # {'status': 'ok'} 나오면 API 정상

In [ ]:
# 2) 고객사 역량 → 맞춤 RFP 추천  (/recommend — 재작성+하이브리드+EXAONE 재랭킹)
import ipywidgets as w
from IPython.display import display, HTML, clear_output

profile = w.Textarea(
    value="공공기관 대상 전자조달·시스템 구축 경험이 많은 SI 기업",
    layout=w.Layout(width="640px", height="70px"))
topk = w.IntSlider(value=5, min=1, max=10, description="추천수")
btn = w.Button(description="맞춤 RFP 추천", button_style="primary")
out = w.Output()

def on_reco(_):
    with out:
        clear_output(); print("검색·EXAONE 재랭킹 중… (수 초~수십 초)")
        res = _post("/recommend", {"profile": profile.value, "top_k": topk.value})
        clear_output()
        html = f"<b>추천 {res['total']}건</b><ol>"
        for it in res["items"]:
            b = f"{it['budget']:,}원" if it.get("budget") else "-"
            html += (f"<li><b>{it.get('title') or it['doc_id']}</b><br>"
                     f"<small>발주 {it.get('org') or '-'} · 예산 {b} · 마감 {it.get('deadline') or '-'} · 관련도 {it.get('score')}</small></li>")
        html += "</ol>"
        display(HTML(html))

btn.on_click(on_reco)
display(w.VBox([w.HTML("<h3>① 고객사 역량 → 맞춤 RFP 추천</h3>"), profile, topk, btn, out]))

In [ ]:
# 3) RFP 질의응답  (/ask — 자체호스팅 EXAONE 생성, 근거 없으면 솔직히 '없다'고 답변)
q = w.Text(value="한국연구재단 기초학문자료센터 시스템 사업의 예산은?",
           layout=w.Layout(width="640px"))
askbtn = w.Button(description="질문하기", button_style="success")
aout = w.Output()

def on_ask(_):
    with aout:
        clear_output(); print("EXAONE 생성 중…")
        res = _post("/ask", {"question": q.value})
        clear_output()
        display(HTML(f"<b>답변</b><br>{res['answer']}"))
        src = "".join(f"<li>{s['source'][:55]} <small>(유사도 {round(s['score'],3)})</small></li>" for s in res["sources"])
        display(HTML(f"<b>출처</b><ul>{src}</ul>"))

askbtn.on_click(on_ask)
display(w.VBox([w.HTML("<h3>② RFP 질의응답 (자체호스팅 EXAONE)</h3>"), q, askbtn, aout]))